Pipeline ภาพรวม

In [299]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [300]:
import glob
import os

red = glob.glob(os.path.join("/content/drive/MyDrive/H3_datanaja/data"))
red

['/content/drive/MyDrive/H3_datanaja/data']

In [301]:
# Paths
BASE_DIR = Path("data")
KB_DIR = BASE_DIR / "knowledge_base"
QUESTIONS_FILE = BASE_DIR / "questions.csv"

#📁 Section 0: Setup & LLM Test

In [302]:
!pip install -q sentence-transformers pythainlp rank-bm25 requests python-dotenv

In [303]:
# import ไลบารี่

## import ไลบารี่

In [304]:
%pip install -q rank_bm25 scikit-learn pandas tqdm


============================================================
### 🎂0 . ปรับ API setup
============================================================

In [305]:
import os
import re
import json
import time
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from collections import Counter

# ============================================================
# API CONFIGURATION
# IMPORTANT: Each model has its own separate endpoint.
# Authentication uses 'apikey' header (NOT 'Authorization: Bearer').
# The model name in the request body is always "/model".
# ============================================================

API_KEY = "zEJ2JyfKmDAguvcUgVndXkNGF7g19vsf"

# Each model has its own base URL under http://thaillm.or.th
MODEL_ENDPOINTS = {
    "openthaigpt": "http://thaillm.or.th/api/openthaigpt/v1",
    "pathumma":    "http://thaillm.or.th/api/pathumma/v1",
    "typhoon":     "http://thaillm.or.th/api/typhoon/v1",
    "kbtg":        "http://thaillm.or.th/api/kbtg/v1",
}

# Human-readable model names (for display)
MODEL_NAMES = {
    "openthaigpt": "OpenThaiGPT-ThaiLLM-8B-instruct-v7.2",
    "pathumma":    "Pathumma-ThaiLLM-qwen3-8b-think-3.0.0",
    "typhoon":     "Typhoon-S-ThaiLLM-8B-Instruct",
    "kbtg":        "THaLLE-0.2-ThaiLLM-8b-fa",
}

# Primary model used for single-model inference
PRIMARY_MODEL = "openthaigpt"

# Request defaults
DEFAULT_MAX_TOKENS = 512
DEFAULT_TEMPERATURE = 0.1
REQUEST_TIMEOUT = 60  # seconds

# Paths
BASE_DIR = Path("data")
KB_DIR = BASE_DIR / "knowledge_base"
QUESTIONS_FILE = BASE_DIR / "questions.csv"
CHECKPOINT_FILE = Path("rag_checkpoint.json")
SUBMISSION_FILE = Path("submission.csv")

print("Configuration loaded.")
print(f"Primary model: {MODEL_NAMES[PRIMARY_MODEL]}")
print(f"Primary endpoint: {MODEL_ENDPOINTS[PRIMARY_MODEL]}")
print(f"Available models: {list(MODEL_ENDPOINTS.keys())}")

Configuration loaded.
Primary model: OpenThaiGPT-ThaiLLM-8B-instruct-v7.2
Primary endpoint: http://thaillm.or.th/api/openthaigpt/v1
Available models: ['openthaigpt', 'pathumma', 'typhoon', 'kbtg']


In [306]:
import os, csv, re, time, requests
import numpy as np
from pathlib import Path
from google.colab import userdata

THAILLM_API_KEY = userdata.get('ThaiLLM')
Pathumm_1a = userdata.get('Pathumm_1a')


In [307]:
!curl http://thaillm.or.th/api/pathumma/v1/chat/completions -H "Content-Type: application/json" -H "apikey: jsakIuGWRcGOwWWXIazfofGUG1hidgof" -d '{"model": "/model", "messages": [{"role": "user", "content": "สวัสดี"}], "max_tokens": 2048, "temperature": 0}'

{"id":"chatcmpl-2eda21a87c12b17b80c46de6efca1272","object":"chat.completion","created":1774739294,"model":"/model","choices":[{"index":0,"message":{"role":"assistant","reasoning_content":null,"content":"<think>\nOkay, the user greeted me with \"สวัสดี\". I should respond in Thai. Let me make sure my response is friendly and appropriate. Maybe say \"สวัสดีค่ะ! มีอะไรให้ช่วยไหมคะ?\" That sounds good. Keep it open-ended so they feel comfortable asking for help.\n</think>\nสวัสดีค่ะ! มีอะไรให้ช่วยไหมคะ? 😊","tool_calls":[]},"logprobs":null,"finish_reason":"stop","stop_reason":null}],"usage":{"prompt_tokens":12,"total_tokens":102,"completion_tokens":90,"prompt_tokens_details":null},"prompt_logprobs":null,"kv_transfer_params":null}

In [308]:
!curl http://thaillm.or.th/api/pathumma/v1/chat/completions -H "Content-Type: application/json" -H "apikey: jsakIuGWRcGOwWWXIazfofGUG1hidgof" -d '{"model": "/model", "messages": [{"role": "user", "content": "สวัสดี"}], "max_tokens": 2048, "temperature": 0.3}'

{"id":"chatcmpl-e1390544e6a03e1595a83a64d473a0ca","object":"chat.completion","created":1774739299,"model":"/model","choices":[{"index":0,"message":{"role":"assistant","reasoning_content":null,"content":"<think>\nOkay, the user greeted me with \"สวัสดี\". I should respond in Thai to keep the conversation friendly. Let me make sure my response is polite and welcoming. Maybe say \"สวัสดีค่ะ! ยินดีที่ได้รับใช้ค่ะ วันนี้มีอะไรให้ช่วยไหมคะ?\" That sounds good. It's friendly and opens the door for them to ask anything they need help with. I'll go with that.\n</think>\nสวัสดีค่ะ! ยินดีที่ได้รับใช้ค่ะ วันนี้มีอะไรให้ช่วยไหมคะ? 😊","tool_calls":[]},"logprobs":null,"finish_reason":"stop","stop_reason":null}],"usage":{"prompt_tokens":12,"total_tokens":158,"completion_tokens":146,"prompt_tokens_details":null},"prompt_logprobs":null,"kv_transfer_params":null}

In [309]:
!curl http://thaillm.or.th/api/typhoon/v1/chat/completions -H "Content-Type: application/json" -H "apikey: jsakIuGWRcGOwWWXIazfofGUG1hidgof" -d '{"model": "/model", "messages": [{"role": "user", "content": "สวัสดี"}], "max_tokens": 2048, "temperature": 0.3}'

{"id":"chatcmpl-5668fe6faf9dcb00abc8468292d77a51","object":"chat.completion","created":1774739308,"model":"/model","choices":[{"index":0,"message":{"role":"assistant","content":"สวัสดีค่ะ! 😊 มีอะไรให้ช่วยไหมคะ?","refusal":null,"annotations":null,"audio":null,"function_call":null,"tool_calls":[],"reasoning":null,"reasoning_content":null},"logprobs":null,"finish_reason":"stop","stop_reason":null,"token_ids":null}],"service_tier":null,"system_fingerprint":null,"usage":{"prompt_tokens":12,"total_tokens":32,"completion_tokens":20,"prompt_tokens_details":null},"prompt_logprobs":null,"prompt_token_ids":null,"kv_transfer_params":null}

### ✅1 : Test the LLM without RAG

Let's ask a FahMai-specific question *without* any context. The LLM shouldn't know the answer.

In [310]:
# Ask without context — LLM has no idea about FahMai's products
response = ask_llm([
    {"role": "user", "content": "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"}
])
print("LLM response (no context):", response)
print("\n→ The LLM doesn't know FahMai-specific facts. We need RAG!")

LLM response (no context): <think>
Okay, the user is asking about the water resistance rating of the Samsung Galaxy S3 Ultra. Let me start by recalling what I know about the S3 Ultra. Wait, actually, the Galaxy S3 Ultra is a variant of the Galaxy S3, but I need to confirm if it's a real model or a different name. Maybe it's a local version or a different carrier's model. Alternatively, perhaps the user is referring to the Galaxy S3 Active, which is a ruggedized version.

First, I should check the official specifications of the Galaxy S3. The standard Galaxy S3 has an IP67 rating, which means it's dust-tight and can withstand immersion in water up to 1 meter for 30 minutes. But the S3 Ultra might have a different rating. If it's the S3 Active, that model has an IP67 rating as well, but it's designed to be more durable against water and dust. However, the user specifically mentioned "Ultra," so maybe it's a different model.

Wait, maybe the user is confused between the S3 Ultra and anoth

### 😎2: Load Knowledge Base

In [355]:
from pathlib import Path

KB_DIR = Path("/content/drive/MyDrive/H3_datanaja/data/knowledge_base") # Define KB_DIR here

documents = []
for fp in sorted(KB_DIR.rglob("*.md")):
    text = fp.read_text(encoding="utf-8")
    documents.append({"path": str(fp.relative_to(KB_DIR)), "text": text})

print(f"Loaded {len(documents)} documents")
print(f"  products/: {sum(1 for d in documents if 'products/' in d['path'])}")
print(f"  policies/: {sum(1 for d in documents if 'policies/' in d['path'])}")
print(f"  store_info/: {sum(1 for d in documents if 'store_info/' in d['path'])}")

# Preview one document only if documents exist
if documents:
    print(f"\n--- Sample: {documents[0]['path']} ---")
    print(documents[0]["text"][:500])
else:
    print("\nNo documents loaded to preview.")

Loaded 118 documents
  products/: 110
  policies/: 5
  store_info/: 3

--- Sample: policies/cancellation_policy.md ---
# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่

**วันที่อัปเดต:** 1 มีนาคม 2569

---

## 1. ภาพรวมนโยบาย

ฟ้าใหม่เข้าใจว่าบางครั้งลูกค้าอาจต้องการยกเลิกคำสั่งซื้อด้วยเหตุผลต่างๆ นโยบายนี้อธิบายสิทธิ์และขั้นตอนการยกเลิกคำสั่งซื้อตามสถานะของคำสั่งซื้อในขณะนั้น ความสามารถในการยกเลิกขึ้นอยู่กับสถานะคำสั่งซื้อเป็นหลัก

---

## 2. การยกเลิกตามสถานะคำสั่งซื้อ

### 2.1 สถานะ "รอชำระเงิน" (Pending Payment)

**ยกเลิกได้ทันที**

คำสั่งซื้อที่อยู่ในสถานะรอชำระเงินสามารถยกเลิกได้ทันทีโดยไม่มีค่าใช้จ่าย ผ่านแอปพลิ


In [356]:
# Update path to use the correct directory in Drive
QUESTIONS_FILE = Path("/content/drive/MyDrive/H3_datanaja/data/questions.csv")

# Load the 100 questions
if QUESTIONS_FILE.exists():
    questions_df = pd.read_csv(QUESTIONS_FILE)
    print(f"Loaded {len(questions_df)} questions.")
    print(f"Columns: {list(questions_df.columns)}")
    print("\nFirst 3 rows:")
    display(questions_df.head(3))
else:
    print(f"Error: File not found at {QUESTIONS_FILE}")

Loaded 100 questions.
Columns: ['id', 'question', 'choice_1', 'choice_2', 'choice_3', 'choice_4', 'choice_5', 'choice_6', 'choice_7', 'choice_8', 'choice_9', 'choice_10']

First 3 rows:


,id,question,choice_1,choice_2,choice_3,choice_4,choice_5,choice_6,choice_7,choice_8,choice_9,choice_10
0,1,Watch S3 Ultra กันน้ำได้กี่ ATM ครับ,3 ATM,IP68,5 ATM,IP67,10 ATM,20 ATM,IPX8,1 ATM,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
1,2,ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ,"2,990 บาท","4,490 บาท","1,990 บาท","4,990 บาท","3,490 บาท","2,490 บาท","3,990 บาท","5,490 บาท",ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
2,3,หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ,Bluetooth 5.0,Bluetooth 5.3,Bluetooth 5.4,Bluetooth 4.2,Bluetooth 5.2,Bluetooth 5.1,Bluetooth 4.0,Bluetooth 5.5,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า


### 😊3: Chunking เเละ Text Chunking & Indexing (TF-IDF + BM25)



In [360]:
from pythainlp.tokenize import word_tokenize

# Demo: tokenize a Thai sentence
sample = "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"
tokens = word_tokenize(sample, engine="newmm")
print(f"Input:  {sample}")
print(f"Tokens: {tokens}")

Input:  Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?
Tokens: ['Watch', ' ', 'S', '3', ' ', 'Ultra', ' ', 'กันน้ำ', 'ได้', 'กี่', ' ', 'ATM', ' ', 'ครับ', '?']


In [329]:
import torch.nn.functional as F

from torch import Tensor
from transformers import AutoTokenizer, AutoModel


def average_pool(last_hidden_states: Tensor,
                 attention_mask: Tensor) -> Tensor:
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

def get_detailed_instruct(task_description: str, query: str) -> str:
    return f'Instruct: {task_description}\nQuery: {query}'

# Each query must come with a one-sentence instruction that describes the task
task = 'Given a web search query, retrieve relevant passages that answer the query'
queries = [
    get_detailed_instruct(task, 'how much protein should a female eat'),
    get_detailed_instruct(task, '南瓜的家常做法')
]
# No need to add instruction for retrieval documents
documents = [
    "As a general guideline, the CDC's average requirement of protein for women ages 19 to 70 is 46 grams per day. But, as you can see from this chart, you'll need to increase that if you're expecting or training for a marathon. Check out the chart below to see how much protein you should be eating each day.",
    "1.清炒南瓜丝 原料:嫩南瓜半个 调料:葱、盐、白糖、鸡精 做法: 1、南瓜用刀薄薄的削去表面一层皮,用勺子刮去瓤 2、擦成细丝(没有擦菜板就用刀慢慢切成细丝) 3、锅烧热放油,入葱花煸出香味 4、入南瓜丝快速翻炒一分钟左右,放盐、一点白糖和鸡精调味出锅 2.香葱炒南瓜 原料:南瓜1只 调料:香葱、蒜末、橄榄油、盐 做法: 1、将南瓜去皮,切成片 2、油锅8成热后,将蒜末放入爆香 3、爆香后,将南瓜片放入,翻炒 4、在翻炒的同时,可以不时地往锅里加水,但不要太多 5、放入盐,炒匀 6、南瓜差不多软和绵了之后,就可以关火 7、撒入香葱,即可出锅"
]
input_texts = queries + documents

tokenizer = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-large-instruct')
model = AutoModel.from_pretrained('intfloat/multilingual-e5-large-instruct')

# Tokenize the input texts
batch_dict = tokenizer(input_texts, max_length=512, padding=True, truncation=True, return_tensors='pt')

outputs = model(**batch_dict)
embeddings = average_pool(outputs.last_hidden_state, batch_dict['attention_mask'])

# normalize embeddings
embeddings = F.normalize(embeddings, p=2, dim=1)
scores = (embeddings[:2] @ embeddings[2:].T) * 100
print(scores.tolist())
# => [[91.92852783203125, 67.580322265625], [70.3814468383789, 92.1330795288086]]


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[[91.9375, 67.625], [70.375, 92.125]]


In [331]:
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi

chunk_texts = [c['text'] for c in chunks]
tokenized_chunks = [simple_thai_tokenize(t) for t in chunk_texts]

bm25_index = BM25Okapi(tokenized_chunks)
tfidf_vectorizer = TfidfVectorizer(tokenizer=simple_thai_tokenize, token_pattern=None, sublinear_tf=True)
tfidf_matrix = tfidf_vectorizer.fit_transform(chunk_texts)
print("Search indexes rebuilt.")

Search indexes rebuilt.


### 📌4: Retrieval  (Keyword + TF-IDF + BM25 Hybrid)






In [332]:
def retrieve_bm25(query: str, top_k: int = 5) -> list[dict]:
    tokens = simple_thai_tokenize(query)
    scores = bm25_index.get_scores(tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = []
    for idx in top_indices:
        if scores[idx] > 0:
            results.append({**chunks[idx], "score": float(scores[idx]), "method": "bm25"})
    return results

def retrieve_tfidf(query: str, top_k: int = 5) -> list[dict]:
    q_vec = tfidf_vectorizer.transform([query])
    sims = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top_indices = np.argsort(sims)[::-1][:top_k]
    results = []
    for idx in top_indices:
        if sims[idx] > 0:
            results.append({**chunks[idx], "score": float(sims[idx]), "method": "tfidf"})
    return results

def hybrid_retrieve(query: str, top_k: int = 6) -> list[dict]:
    """
    Combines BM25 and TF-IDF for robust keyword-based retrieval.
    """
    bm25_res = retrieve_bm25(query, top_k=top_k)
    tfidf_res = retrieve_tfidf(query, top_k=top_k)

    scored = {}
    def _add(res_list, weight):
        if not res_list: return
        max_s = max(r['score'] for r in res_list) or 1.0
        for r in res_list:
            cid = r['chunk_id']
            norm_score = r['score'] / max_s
            if cid not in scored:
                scored[cid] = {**r, 'hybrid_score': 0.0}
            scored[cid]['hybrid_score'] += weight * norm_score

    _add(bm25_res, 0.6)
    _add(tfidf_res, 0.4)

    ranked = sorted(scored.values(), key=lambda x: x['hybrid_score'], reverse=True)
    return ranked[:top_k]

### 🔑5 .Generate Answer

Now, let's combine the retrieved context with the user's question and send it to the LLM to generate a coherent answer.

In [334]:
# A very basic system prompt — you should improve this!
SYSTEM_PROMPT = "ĕġāĐāġāāĠŁłıİĠĠŁłıĠı đđłıİıĠŁłıĠı đđłıİıĠŁłıĠı đđłıİıĠŁłıĠı ANSWER: X đđłıİıĠŁłıĠı"

def build_rag_prompt(question, choices, retrieved_chunks):
    context = "\n\n".join(
        f"--- Chunk {i+1} ---\n{c['text']}"
        for i, c in enumerate(retrieved_chunks)
    )
    choices_text = "\n".join(f"{k}. {v}" for k, v in choices.items())
    return (
        f"{context}\n\n"
        f"đđłıİı: {question}\n\n"
        f"đđłıİı:\n{choices_text}\n\n"
        f"đđłıİı ANSWER: X (X đđłıİı 1-10)"
    )

# Demo: answer Q1
q = questions[0]
# Changed to use the hybrid_retrieve function instead of the missing dense_retrieve
retrieved = hybrid_retrieve(q["question"], top_k=3)

prompt = build_rag_prompt(q["question"], q["choices"], retrieved)

# Call the LLM using requests
url = MODEL_ENDPOINTS[PRIMARY_MODEL] + "/chat/completions"
headers = {"Content-Type": "application/json", "apikey": Pathumm_1a}
data = {
    "model": "/model",
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ],
    "max_tokens": 512,
    "temperature": 0
}

response = requests.post(url, headers=headers, json=data)
if response.status_code == 200:
    raw = response.json()['choices'][0]['message']['content']
    # Clean up reasoning if present
    clean_raw = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
    print(f"đđłıİı: {q['question']}")
    print(f"LLM đđłıİı: {clean_raw}")
else:
    print(f"Error: {response.status_code}")

đđłıİı: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
LLM đđłıİı: Watch S3 Ultra กันน้ำได้ 10 ATM ครับ

**คำอธิบายเพิ่มเติม:**

ATM ย่อมาจาก "Atmosphere" ซึ่งเป็นหน่วยที่ใช้ในการวัดความดัน สำหรับนาฬิกาสมาร์ทวอทช์ ค่า ATM นี้บ่งบอกถึงความสามารถในการกันน้ำของนาฬิกา โดยทั่วไปแล้ว นาฬิกาที่กันน้ำได้ 1 ATM สามารถทนต่อการกันน้ำได้ในระดับน้ำลึกประมาณ 10 เมตร แต่ไม่แนะนำให้ใช้ในกิจกรรมที่เกี่ยวข้องกับน้ำ เช่น ว่ายน้ำ หรือดำน้ำ

นาฬิกาที่กันน้ำได้ 5 ATM สามารถทนต่อการกันน้ำได้ในระดับน้ำลึกประมาณ 50 เมตร ซึ่งเหมาะสำหรับการว่ายน้ำในสระหรือทะเล แต่ไม่เหมาะสำหรับการดำน้ำลึก

นาฬิกาที่กันน้ำได


### 😆6: Prompt Engineering (Thai Prompts, Few-shot Examples)

In [335]:
SYSTEM_PROMPT = """คุณเป็นผู้เชี่ยวชาญการวิเคราะห์ข้อมูลของร้าน FahMai (ฟ้าใหม่)
หน้าที่ของคุณคือตอบคำถามปรนัย (Multiple Choice) โดยใช้ข้อมูลจาก [ฐานข้อมูล] ที่กำหนดให้เท่านั้น

กฎข้อบังคับ:
1. ตรวจสอบข้อมูลใน [ฐานข้อมูล] อย่างละเอียดก่อนตอบ
2. เลือกตัวเลข (1-10) ที่ตรงกับตัวเลือกที่ถูกต้องที่สุดเพียงตัวเดียว
3. หากไม่มีข้อมูลในฐานข้อมูลที่ระบุคำตอบได้ ให้ตอบว่า '9'
4. หากคำถามไม่เกี่ยวกับสินค้า นโยบาย หรือบริการของร้าน FahMai ให้ตอบว่า '10'
5. ห้ามเขียนอธิบาย ห้ามแสดงความคิดเห็น ให้ตอบเป็น 'ตัวเลข' เพียงอย่างเดียวเท่านั้น"""

def build_rag_prompt(question, choices, context_chunks):
    context_str = "\n".join([f"- {c['text']}" for c in context_chunks])
    # Handle list or dict choices
    if isinstance(choices, dict):
        choices_str = "\n".join([f"{k}. {v}" for k, v in choices.items()])
    else:
        choices_str = "\n".join([f"{i+1}. {c}" for i, c in enumerate(choices)])

    user_message = f"""[ฐานข้อมูล]
{context_str}

[คำถาม]
{question}

[ตัวเลือก]
{choices_str}

ตอบเฉพาะหมายเลขตัวเลือก (1-10):"""
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_message}]

### 😺 7: LLM Inference Functions

Using the `requests` library directly because the API uses a non-standard `apikey` header (not `Authorization: Bearer`).

In [336]:
def parse_answer(raw_text: str) -> int:
    """
    Strictly extracts only the choice number 1-10.
    Ignores reasoning, thoughts, or explanations.
    """
    raw_text = strip_think_tags(raw_text).strip()

    # Look for patterns like "ตอบ: 5", "Choice: 5", or just "5"
    # Search for 10 first to avoid matching '1' in '10'
    match = re.search(r'\b(10|[1-9])\b', raw_text)
    if match:
        return int(match.group(1))

    # Fallback: check if the string contains any number
    digits = re.findall(r'\d+', raw_text)
    if digits:
        val = int(digits[0])
        return val if 1 <= val <= 10 else 9

    return 9 # Default to 'No information' if parsing fails

In [337]:
# ============================================================
# Quick API connectivity test
# ============================================================

def test_api_connection(model_key: str = PRIMARY_MODEL) -> bool:
    """Send a simple greeting to verify the API is reachable."""
    print(f"Testing connection to model: {MODEL_NAMES[model_key]}")
    print(f"Endpoint: {MODEL_ENDPOINTS[model_key]}")
    try:
        response = call_llm(
            messages=[{"role": "user", "content": "สวัสดี ตอบสั้นๆ ด้วย"}],
            model_key=model_key,
            max_tokens=50,
            temperature=0.3,
            retries=2,
        )
        if response:
            print(f"SUCCESS — response: {response[:100]}")
            return True
        else:
            print("FAILED — empty response")
            return False
    except Exception as e:
        print(f"FAILED — {e}")
        return False


# Run test
test_api_connection(PRIMARY_MODEL)

Testing connection to model: OpenThaiGPT-ThaiLLM-8B-instruct-v7.2
Endpoint: http://thaillm.or.th/api/openthaigpt/v1
SUCCESS — response: <think>
ผู้ใช้ต้องการให้ตอบด้วยภาษาไทยเป็นหลัก และสามารถใช้ภาษาอื่นได้บ้าง แต่ต้องไม่น้อยกว่า 150 คำ


True

### ✅8: Main RAG Pipeline (with Checkpoint/Resume)

In [338]:
def get_question_columns(df: pd.DataFrame) -> tuple[str, list[str]]:
    """
    Auto-detect the question column and choice columns from the DataFrame.
    Returns (question_col, [choice_cols]).
    """
    cols = list(df.columns)

    # Detect question column
    question_col = None
    for candidate in ["question", "คำถาม", "Question", "q"]:
        if candidate in cols:
            question_col = candidate
            break
    if question_col is None:
        # Guess: first non-id string column
        for c in cols:
            if c.lower() not in ("id", "answer") and df[c].dtype == object:
                question_col = c
                break

    # Detect choice columns (choice_1..choice_10 or choice1..choice10)
    choice_cols = [c for c in cols if re.match(r'choice[_\s]?\d+', c, re.IGNORECASE)]
    choice_cols.sort(key=lambda x: int(re.search(r'\d+', x).group()))

    if not choice_cols:
        # Fallback: look for columns named 1-10 or option_1..10
        choice_cols = [c for c in cols if re.match(r'(option|opt|ans|choice)?[_\s]?\d+$', c, re.IGNORECASE) and c not in (question_col, 'id', 'answer')]
        choice_cols.sort(key=lambda x: int(re.search(r'\d+', x).group()))

    return question_col, choice_cols


question_col, choice_cols = get_question_columns(questions_df)
print(f"Question column: '{question_col}'")
print(f"Choice columns ({len(choice_cols)}): {choice_cols}")

Question column: 'question'
Choice columns (10): ['choice_1', 'choice_2', 'choice_3', 'choice_4', 'choice_5', 'choice_6', 'choice_7', 'choice_8', 'choice_9', 'choice_10']


#🙏9: Multi-Model Ensemble

In [358]:
def run_ensemble_pipeline(df, models=["openthaigpt", "typhoon", "pathumma"]):
    final_results = {}
    all_votes = {str(row['id']): [] for _, row in df.iterrows()}

    for m_key in models:
        print(f"\n--- Running Inference with {m_key} ---")
        # Use a temporary checkpoint for each model to avoid mixing
        res = run_rag_pipeline(df, model_key=m_key, force_rerun=True)
        for qid, val in res.items():
            all_votes[qid].append(val['answer'])

    print("\n--- Calculating Final Ensemble Results (Majority Vote) ---")
    for qid, votes in all_votes.items():
        # Majority vote logic
        counts = Counter(votes)
        best_answer = counts.most_common(1)[0][0]
        final_results[qid] = {"answer": best_answer, "votes": votes}

    return final_results

# Pipeline definition (maintained from previous for consistency)
def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f: return json.load(f)
    return {}

def save_checkpoint(results):
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f: json.dump(results, f, ensure_ascii=False)

def run_rag_pipeline(df, model_key=PRIMARY_MODEL, top_k=6, sleep_between=0.5, force_rerun=False):
    results = {} if force_rerun else load_checkpoint()
    q_col, c_cols = get_question_columns(df)
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"RAG [{model_key}]"):
        qid = str(row["id"])
        if qid in results: continue
        question, choices = str(row[q_col]), [str(row[c]) for c in c_cols if c in row.index]
        context_chunks = hybrid_retrieve(question, top_k=top_k)
        messages = build_rag_prompt(question, choices, context_chunks)
        raw = call_llm(messages, model_key=model_key)
        answer = parse_answer(raw)
        results[qid] = {"answer": answer, "model": model_key}
        save_checkpoint(results)
        time.sleep(sleep_between)
    return results

In [359]:
# Execute full pipeline using the Ground Truth logic and improved hybrid retrieval
single_model_results = run_rag_pipeline(questions_df, model_key="typhoon", force_rerun=True)

# Quick validation of the prediction distribution
ans_list = [v['answer'] for v in single_model_results.values()]
print("\nFinal Smart Prediction Distribution:")
import pandas as pd
display(pd.Series(ans_list).value_counts().sort_index())

RAG [typhoon]:  44%|████▍     | 44/100 [04:59<04:43,  5.07s/it]

[WARN] HTTP error 502 on attempt 1/3: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/typhoon/v1/chat/completions


RAG [typhoon]: 100%|██████████| 100/100 [08:45<00:00,  5.25s/it]


Final Smart Prediction Distribution:


,count
1,9
2,9
3,9
4,14
5,7
6,7
7,7
8,4
9,33
10,1


### 📁 10: Generate Submission


In [361]:
def generate_submission(
    results: dict,
    df: pd.DataFrame,
    output_path: Path = SUBMISSION_FILE,
    use_ensemble: bool = False,
) -> pd.DataFrame:
    """
    Generate submission.csv from pipeline results.
    """
    rows = []
    for _, row in df.iterrows():
        qid = str(row["id"])
        if use_ensemble:
            answer = results.get(qid, 9)
        else:
            # single_model_results structure is {qid: {'answer': val, ...}}
            answer = results.get(qid, {}).get("answer", 9)
        rows.append({"id": row["id"], "answer": answer})

    submission_df = pd.DataFrame(rows)[["id", "answer"]]
    submission_df.to_csv(output_path, index=False)
    print(f"Submission saved to: {output_path}")
    print(f"Total rows: {len(submission_df)}")
    return submission_df

# Generate from the latest results
submission_df = generate_submission(
    results=single_model_results,
    df=questions_df,
    output_path=SUBMISSION_FILE,
    use_ensemble=False,
)

print("\nSubmission preview:")
display(submission_df.head(10))

print("\nAnswer distribution:")
display(submission_df["answer"].value_counts().sort_index())

Submission saved to: submission.csv
Total rows: 100

Submission preview:


,id,answer
0,1,5
1,2,7
2,3,1
3,4,9
4,5,9
5,6,9
6,7,9
7,8,4
8,9,4
9,10,2



Answer distribution:


,count
answer,
1,9
2,9
3,9
4,14
5,7
6,7
7,7
8,4
9,33


In [342]:
# Validate submission format
assert list(submission_df.columns) == ["id", "answer"], "Columns must be ['id', 'answer']"
assert len(submission_df) == len(questions_df), f"Expected {len(questions_df)} rows, got {len(submission_df)}"
assert submission_df["answer"].between(1, 10).all(), "All answers must be between 1 and 10"
assert not submission_df["id"].duplicated().any(), "Duplicate IDs found!"

print("Submission validation passed!")
print(f"  Rows: {len(submission_df)}")
print(f"  Answer range: {submission_df['answer'].min()} – {submission_df['answer'].max()}")

Submission validation passed!
  Rows: 100
  Answer range: 1 – 10


## 😺 11: Analysis


In [343]:
def analyze_results(results: dict, df: pd.DataFrame) -> None:
    """
    Print an analysis of the pipeline results:
    - Answer distribution
    - Empty/failed responses
    - Most frequently retrieved documents
    """
    q_col, _ = get_question_columns(df)
    answered = [v for v in results.values() if isinstance(v, dict)]

    print(f"Total answered: {len(answered)}/{len(df)}")
    empty = [v for v in answered if not v.get("raw")]
    print(f"Empty/failed responses: {len(empty)}")

    answer_counts = Counter(v["answer"] for v in answered)
    print("\nAnswer distribution:")
    for i in range(1, 11):
        label = ""
        if i == 9:  label = " ← ไม่มีข้อมูลในฐานข้อมูล"
        if i == 10: label = " ← คำถามไม่เกี่ยวข้อง"
        print(f"  {i:2d}: {answer_counts.get(i, 0):3d} times{label}")

    # Most retrieved documents
    all_files = []
    for v in answered:
        all_files.extend(v.get("context_files", []))
    print("\nTop 10 most retrieved documents:")
    for fname, cnt in Counter(all_files).most_common(10):
        print(f"  {fname}: {cnt} times")


analyze_results(single_model_results, questions_df)

Total answered: 100/100
Empty/failed responses: 100

Answer distribution:
   1:  13 times
   2:  10 times
   3:   6 times
   4:  13 times
   5:   7 times
   6:   6 times
   7:   7 times
   8:   5 times
   9:  32 times ← ไม่มีข้อมูลในฐานข้อมูล
  10:   1 times ← คำถามไม่เกี่ยวข้อง

Top 10 most retrieved documents:


In [344]:
def debug_question(qid: str, results: dict, df: pd.DataFrame, top_k: int = 6) -> None:
    """
    Show full details for a single question: context retrieved, prompt, raw LLM response, answer.
    """
    q_col, c_cols = get_question_columns(df)

    row = df[df["id"].astype(str) == str(qid)]
    if row.empty:
        print(f"Question id '{qid}' not found.")
        return

    row = row.iloc[0]
    question = str(row[q_col])
    choices  = [str(row[c]) for c in c_cols if c in row.index]

    print(f"{'='*70}")
    print(f"Question ID: {qid}")
    print(f"Question: {question}")
    print("\nChoices:")
    for i, c in enumerate(choices, 1):
        print(f"  {i}. {c}")

    # Retrieve context
    context = hybrid_retrieve(question, top_k=top_k)
    print(f"\nRetrieved {len(context)} context chunks:")
    for i, chunk in enumerate(context, 1):
        # Changed 'filename' -> 'path' and 'chunk_text' -> 'text'
        print(f"\n  [{i}] {chunk.get('path', 'unknown')} (score={chunk.get('hybrid_score', 0.0):.3f})")
        print(f"  {chunk.get('text', '')[:200]}...")

    # Show stored result
    stored = results.get(str(qid))
    if stored:
        print(f"\nModel key: {stored.get('model', 'unknown')}")
        print(f"Parsed answer: {stored.get('answer')}")
    else:
        print("\nNo stored result for this question.")


# Debug example: change the qid to inspect any question
if questions_df is not None and len(questions_df) > 0:
    sample_qid = str(questions_df.iloc[0]["id"])
    debug_question(sample_qid, single_model_results, questions_df)

Question ID: 1
Question: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

Choices:
  1. 3 ATM
  2. IP68
  3. 5 ATM
  4. IP67
  5. 10 ATM
  6. 20 ATM
  7. IPX8
  8. 1 ATM
  9. ไม่มีข้อมูลนี้ในฐานข้อมูล
  10. คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า

Retrieved 6 context chunks:

  [1] /content/drive/MyDrive/H3_datanaja/data/knowledge_base/products/WK-SW-001_wongkhojon_watch_s3_ultra.md (score=0.964)
  าร์จแม่เหล็กของ S3 Ultra ใช้ร่วมกับ Watch S3 Pro ได้เลย เพราะใช้มาตรฐานเดียวกัน...

  [2] /content/drive/MyDrive/H3_datanaja/data/knowledge_base/products/WK-SW-001_wongkhojon_watch_s3_ultra.md (score=0.920)
  # วงโคจร Watch S3 Ultra (WongKhoJon Watch S3 Ultra)

รหัสสินค้า: WK-SW-001
แบรนด์: วงโคจร (WongKhoJon) — แบรนด์ในเครือฟ้าใหม่
หมวดหมู่: สมาร์ทวอทช์
ราคา: ฿14,990
สถานะ: มีสินค้า
วันที่อัปเดต: 1 มีนาคม...

  [3] /content/drive/MyDrive/H3_datanaja/data/knowledge_base/products/WK-SW-003_wongkhojon_watch_s3.md (score=0.810)
  # วงโคจร Watch S3 (WongKhoJon Watch S3)

รหัสสินค้า: WK-SW-003
แบรนด์: วงโคจ